In [1]:
# 環境架設
from selenium import webdriver
from selenium.webdriver.remote.webelement import WebElement
# driver = webdriver.Chrome()
# driver = webdriver.Firefox()
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
# pip install webdriver-manager
# driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))

import xpon_module as xpon
import selenium_module as mod
import xpon_olt_status_module as olts
import xpon_olt_config_module as oltc
import xpon_batch_ont_config_module as bat
import xpon_ont_config_module as ontc
import xpon_profiles_module as pro
# import multicast_module as mul

# copy from GLC project-------------
import acl_module as acl  
import glc_test_module as glc
# ----------------------------------

from selenium.webdriver.support.ui import Select
import time
import datetime
import pyautogui
from docx import Document
from docx.shared import Inches

import ems_acl_module as e_acl
import ems_profile_module as e_pro
import ems_test_module as ems

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from customdriver import CustomDriver
import globals
import get_card

# import videocap_module as videocap

time_sleep_before_find_element = 0 # 0.1
driver = CustomDriver(time_sleep = time_sleep_before_find_element)

# 讀取ENV.txt檔
set_up_file = open("ENV.txt",'r')
lines = set_up_file.readlines()
set_up_file.close()
url = lines[0].split("=")[1]
url_ip = url.split("/")[2] 
username = lines[1].split("=")[1]
password = lines[2].split("=")[1]
loop = int(lines[3].split("'")[1])

#Card
msc_slot = lines[6].split("'")[1]
msc_port = lines[7].split("'")[1]
glc_slot = lines[8].split("'")[1]
glc_port = lines[9].split("'")[1]

#Gpon
slot_1_aid = lines[12].split("'")[1]
pon_1_aid = lines[13].split("'")[1]
ont_1_1 = lines[16].split("'")[1]
ont_1_2 = lines[17].split("'")[1]
ont_1_1_aid = lines[19].split("'")[1]
ont_1_2_aid = lines[20].split("'")[1]

slot_2_aid = lines[24].split("'")[1]
pon_2_aid = lines[25].split("'")[1]
ont_2_1 = lines[28].split("'")[1]
ont_2_2 = lines[29].split("'")[1]
ont_2_1_aid = lines[31].split("'")[1]
ont_2_2_aid = lines[32].split("'")[1]

ont_1_1_sn = lines[36].split("'")[1]
ont_1_1_pw = lines[37].split("'")[1]
ont_1_1_ont_USER = lines[38].split("=")[1]
ont_1_1_ont_PW = lines[39].split("=")[1]

ont_1_2_sn = lines[41].split("'")[1]
ont_1_2_pw = lines[42].split("'")[1]
ont_1_2_ont_USER = lines[43].split("=")[1]
ont_1_2_ont_PW = lines[44].split("=")[1]

ont_2_1_sn = lines[46].split("'")[1]
ont_2_1_pw = lines[47].split("'")[1]
ont_2_1_ont_USER = lines[48].split("=")[1]
ont_2_1_ont_PW = lines[49].split("=")[1]

ont_2_2_sn = lines[51].split("'")[1]
ont_2_2_pw = lines[52].split("'")[1]
ont_2_2_ont_USER = lines[53].split("=")[1]
ont_2_2_ont_PW = lines[54].split("=")[1]

#Xpon
slot_x_1_aid = lines[58].split("'")[1]
pon_x_1_aid = lines[59].split("'")[1]
ont_x_1_1 = lines[62].split("'")[1]
ont_x_1_2 = lines[63].split("'")[1]
ont_x_1_1_aid = lines[65].split("'")[1]
ont_x_1_2_aid = lines[66].split("'")[1]

slot_x_2_aid = lines[70].split("'")[1]
pon_x_2_aid = lines[71].split("'")[1]
ont_x_2_1 = lines[74].split("'")[1]
ont_x_2_2 = lines[75].split("'")[1]
ont_x_2_1_aid = lines[77].split("'")[1]
ont_x_2_2_aid = lines[78].split("'")[1]

ont_x_1_1_sn = lines[82].split("'")[1]
ont_x_1_1_pw = lines[83].split("'")[1]
ont_x_1_1_ont_USER = lines[84].split("=")[1]
ont_x_1_1_ont_PW = lines[85].split("=")[1]

ont_x_1_2_sn = lines[87].split("'")[1]
ont_x_1_2_pw = lines[88].split("'")[1]
ont_x_1_2_ont_USER = lines[89].split("=")[1]


ont_x_1_2_ont_PW = lines[90].split("=")[1]

ont_x_2_1_sn = lines[92].split("'")[1]
ont_x_2_1_pw = lines[93].split("'")[1]
ont_x_2_1_ont_USER = lines[94].split("=")[1]
ont_x_2_1_ont_PW = lines[95].split("=")[1]

ont_x_2_2_sn = lines[97].split("'")[1]
ont_x_2_2_pw = lines[98].split("'")[1]
ont_x_2_2_ont_USER = lines[99].split("=")[1]
ont_x_2_2_ont_PW = lines[100].split("=")[1]

# service_mode_x_1 = lines[104].split("'")[1]
service_mode_x_1 = ''

# snmp_get = lines[110].split("=")[1]
snmp_get = ''
snmp_set = '' #lines[111].split("=")[1]
snmp_version = '' #lines[112].split("=")[1]

ont_1_1 = int(ont_1_1)-1
ont_1_2 = int(ont_1_2)-1
ont_2_1 = int(ont_2_1)-1
ont_2_2 = int(ont_2_2)-1

ems_password = "Pqa1234!!"
test_retry_num = 3

start_time = time.time()




#
#    FW Web CardType Case       
# ====================================

driver.get(url) # 連上網頁
mod.Login(driver,username,password) # 登入
 
# FW get msc_card_type, msc_version
mod.Switch_Default(driver)
mod.Page_Switch(driver,"Home")
time.sleep(5)
mod.Switch_Frame(driver)
msc_card_type = driver.find_elements('class name',"td-table-content.SysinfoCardType.left-wrapper")[int(msc_slot)-1].get_attribute("textContent")
globals.msc_card_type = msc_card_type # test
msc_version = driver.find_elements('class name',"td-table-content.Sysinfofwversion.left-wrapper")[int(msc_slot)-1].get_attribute("textContent")

# Config Reset ---
# ems.Maintenance_Config_Reset(driver)  #*自動化全程都不能點其他東西，測試頁面必須在最前面

# FW Sys - Access Control - SNMP, input value from ENV.txt ---
# FW Sys - User Account, input value from ENV.txt ---
ems.EMS_Set_Prepare(driver,snmp_get,snmp_set,snmp_version,username,password)

# cardtype case
list1 = get_card.get_card_case(driver,glc_port,pon_x_1_aid,pon_1_aid,file_name='')

# kick?
# driver.quit()
ems.FW_Logout(driver)


#    EMS Web        
# ====================================
# driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
# driver = CustomDriver(time_sleep = time_sleep_before_find_element)
driver.get("https://192.168.128.8:9116/") 
time.sleep(5)
mod.Login_EMS(driver,username,ems_password)
time.sleep(5)
try:
    driver.find_element('id','helpMenu').click()
except:
    mod.Login_EMS(driver,username,ems_password)
    time.sleep(5)
    driver.find_element('id','helpMenu').click()
driver.find_element('id','about_ems_button').click()
time.sleep(3)
emsversion = driver.find_element('id','emsversion').text.split("\n")[0]
time.sleep(5)
driver.find_element('xpath', '//*[@type="button" and @class="button button_green"]').click()    ### click close

#    Set File Name
date1 = str(datetime.datetime.now()).split(":")[0]
date2 = str(datetime.datetime.now()).split(":")[1]
file_name = f"Output\Web_EMS_{emsversion}_{date1}{date2}.txt"
# file_name = f"Output\Web_EMS_test.txt"

#    Info in File
test_report = open(file_name,"w+")
print("EMS Web Auto Report",file = open(file_name, '+a'))
print(datetime.datetime.now(),file = open(file_name, '+a'))
print(f"EMS Web IP:http://192.168.128.8:9116/  \nEMS Version:{emsversion} ",file = open(file_name, '+a'))
print(f"FW Web IP:{url}FW MSC:{msc_slot}/{msc_card_type}/{msc_version}",file = open(file_name, '+a'))
print(f"FW Web IP:{url}FW MSC:{msc_slot}-{msc_port}/{msc_card_type}/{msc_version}",file = open(file_name, '+a'))
print(f"{list1}",file = open(file_name, '+a'))
get_card.print_case(list1,file_name)
MSC_slot = list1.pop()
MSC_card = list1.pop()
print(f"  (find_element(), find_elements() will wait {time_sleep_before_find_element} sec )",file = open(file_name, '+a'))
print("======================================================================================",file = open(file_name, '+a'))

print(f"EMS Test\nTest Start",file = open(file_name, '+a')) 


dict1:{'GLC1440X-55A': '1', 'MSC1240XA': '9', 'GLC1540X-55A': '15', 'OLC3416B-IA': '17'}
dict_keys(['GLC1440X-55A', 'MSC1240XA', 'GLC1540X-55A', 'OLC3416B-IA'])
slot:1 GLC1440X-55A, in case (1) GLC1440X, GLC1540X
slot:9 MSC1240XA, in case (4) MSC
slot:15 GLC1540X-55A, in case (1) GLC1440X, GLC1540X
slot:17 OLC3416B-IA, in case (2) OLC3416B, OLC3708, OLC3816
list1:['1,1,39,GLC1440X-55A', '4,9,16,MSC1240XA', '1,15,39,GLC1540X-55A', '2,17,16,OLC3416B-IA']
list1:['1,1,39,GLC1440X-55A', '4,9,16,MSC1240XA', '1,15,39,GLC1540X-55A', '2,17,16,OLC3416B-IA', 'MSC1240XA', '9']
             [Statistics - Online Users] session:*http-3


In [2]:
# # #    Provision > Profile 
# # # ====================================

# print(f"EMS - Profile Test Start\n",file = open(file_name, '+a'))

# ems.EMS_delete_profiles(driver,username,ems_password,file_name)
# ems.EMS_test_all_Profile(driver,url,list1,MSC_card,MSC_slot,username,ems_password,file_name,  2)



# # # ems.IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)           # -V

# # # ems.ONT_Service_Profile(driver,username,ems_password,file_name)              # V
# # # ems.ONT_QoS_Bandwidth_Profile(driver,username,ems_password,file_name)      # V
# # # ems.ONT_Multicast_Profile(driver,username,ems_password,file_name)          # -V
# # # ems.ONT_UNI_Profile(driver,username,ems_password,file_name)                # -V
# # # ems.ONT_Security_Profile(driver,username,ems_password,file_name)           # V
# # ems.ONT_ONT_Profile(driver,username,ems_password,file_name)                # -V
# # # ems.ONT_VoIP_SIP_Profile(driver,username,ems_password,file_name)           # -V
# # # ems.ONT_VoIP_Common_Profile(driver,username,ems_password,file_name)        # V
# # # ems.ONT_VoIP_Dial_Plan_Profile(driver,username,ems_password,file_name)     # V
# # # ems.ONT_ACL_Profile(driver,username,ems_password,file_name)                # V 
# # # ems.ONT_Alarm_Profile(driver,username,ems_password,file_name)              # V

# # # ems.ONT_Template_Profile(driver,username,ems_password,file_name)           # X need add create other profile  


# # # ems.Shaping_Profile(driver,username,ems_password,file_name)                        # -V  
# # # ems.Weight_Profile(driver,username,ems_password,file_name)                         # -V
# # # ems.Rate_Limit_Profile(driver,username,ems_password,file_name)                     # -V
# # # ems.GE_Template(driver,username,ems_password,file_name)                            # X  need add create other profile


# print(f"EMS - Profile Test End\n",file = open(file_name, '+a'))

# end_time = time.time() 
# total_time = end_time - start_time
# print(f'Profile Test   Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))
# # # ->>> Out of Memory occured!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [3]:
# Card Test
list1_o = ['4,9,16,MSC1240XA', '1,15,39,GLC1540X-55A', '2,17,16,OLC3416B-IA']
list1 = '1,15,39,GLC1540X-55A'


In [3]:
ems.Port_DHCP(driver,url,list1,username,password,file_name)

In [4]:
from ems_test_module import *

In [4]:
ems.VLAN_VLAN_Table(driver,url,list1,username,password,file_name) 

index:4
index:4


In [6]:
ems.Switch_Storm_Control(driver,url,list1,username,password,file_name)

In [5]:
from ems_test_module import *

In [4]:
# card
ems.Multicast_Multicast_MAC(driver,url,list1,username,password,file_name)                 # case
ems.Multicast_Multicast_IP(driver,url,list1,username,password,file_name)                    #   case



In [5]:
# no card
ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name)      # msc


In [ ]:
# card NEED STEP CHECK !!
ems.EMS_ACL_MAC_Count(driver,url,list1,username,password,file_name)   
ems.Port_DHCP(driver,url,list1,username,password,file_name)                     # case

In [2]:
# # # No Card (173min)
ems.EMS_ACL_DHCP_Remote_Backup(driver,url,MSC_card,MSC_slot,username,password,file_name)  #   msc
ems.EMS_ACL_PPPoE(driver,url,MSC_card,username,password,file_name)                        #   msc  
ems.EMS_ACL_Multicast(driver,url,MSC_card,username,password,file_name)                    #   msc
ems.Multicast_IGMP_Setup(driver,url,MSC_card,username,password,file_name)                 # msc need_modify   X
# ems.Multicast_VLAN_Mode(driver,url,MSC_card,username,password,file_name)                  # msc need_modify   X
# ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name)      # msc
# # (Sys)                                                                                     # msc                                                                               
ems.Sys_General_Setup(driver,url,MSC_card,username,password,file_name) 
Ip_setup_gateway = ems.Sys_IPv4_Setup(driver,url,MSC_card,username,password,file_name)
ems.Sys_IPv6_Setup(driver,url,MSC_card,username,password,file_name)
ems.Sys_IP_Route(driver,url,MSC_card,username,password,file_name, Ip_setup_gateway)    
ems.Switch_General(driver,url,MSC_card,username,password,file_name)                         # msc 
ems.Switch_Dot3ad(driver,url,MSC_card,MSC_slot,msc_port,username,password,file_name)        # msc   undone
ems.Switch_Isolation(driver,url,MSC_card,username,password,file_name)                       # msc 
ems.VLAN_Tag_Mode_Setup(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc


In [3]:
ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name) 
ems.Multicast_VLAN_Mode(driver,url,MSC_card,username,password,file_name)                  # msc need_modify   X

In [4]:
from ems_test_module import *

In [24]:
driver.get(url) 

In [22]:
EMS_Provision_Config_path(driver,url)
driver.find_element(By.LINK_TEXT,"Multicast").click()
driver.find_element(By.LINK_TEXT,"MVLAN Table").click()

In [19]:
# driver.find_element(By.XPATH,'//*[@for="multicast_mvlantable_radio2"]')
driver.find_element(By.XPATH,'//*[@id="multicast_mvlantable_table"]/div[3]/div[3]/label')

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="multicast_mvlantable_table"]/div[3]/div[3]/label"}
  (Session info: chrome=122.0.6261.112)
Stacktrace:
	GetHandleVerifier [0x00D4E123+48179]
	(No symbol) [0x00CD5D01]
	(No symbol) [0x00BBD72D]
	(No symbol) [0x00BF572B]
	(No symbol) [0x00BF586B]
	(No symbol) [0x00C2C5B2]
	(No symbol) [0x00C13CD4]
	(No symbol) [0x00C2AA1E]
	(No symbol) [0x00C13A26]
	(No symbol) [0x00BEB7BC]
	(No symbol) [0x00BEC62D]
	GetHandleVerifier [0x01067C33+3299139]
	GetHandleVerifier [0x010A5BF2+3553026]
	GetHandleVerifier [0x010A0BCC+3532508]
	GetHandleVerifier [0x00DE494E+664670]
	(No symbol) [0x00CE0AB4]
	(No symbol) [0x00CDBF08]
	(No symbol) [0x00CDC02D]
	(No symbol) [0x00CCDD00]
	BaseThreadInitThunk [0x76A900C9+25]
	RtlGetAppContainerNamedObjectPath [0x77447B4E+286]
	RtlGetAppContainerNamedObjectPath [0x77447B1E+238]


In [23]:
sel = '2'
e_mod.Checkbox_EMS(driver,f"multicast_mvlantable_radio2",True)
# //*[@for="multicast_mvlantable_radio2"]

In [4]:
ems.Port_Qos(driver,url,list1,username,password,file_name)

In [ ]:
driver.find_element(By.XPATH,'//*[@id="message_diag"]/div/div/div[3]/button').click()

In [ ]:
from ems_test_module import *

In [ ]:
list1

In [ ]:
# EMS msc should not support Test ... 

# ACL - Anti MAC Spoof
# ACL - MAC Count   
# ACL - MAC Filter  
# ACL - OUI Filter
# ACL - 802.1x - Port Setting
# ACL - Packet Filter
# ACL - ACL Egress Rule
# ACL - MACFF - Apply Table
# ACL - MACFF - Arp Proxy

# Diagnostic - Line Diagnostic - DELT
# Diagnostic - Line Diagnostic - SELT
# Diagnostic - OAM Loopback

# Line Card - G.vector
# Line Card - PIN
# Multicast - IGMP - Port MVLAN Setup #?
# Multicast - IGMP - Port Mode # -> Multicast - Port Mode
# Multicast - Static Multicast - Multicast IP # -> Multicast - Multicast IP
# Port - DHCP
# Port - DSL
# Port - GENNI
# Port - GE
# Port - FDB
# Port - G.bond
# Port - PPPoE
# Port - PVC
# Port - Power-saving
# Statistics - DSL Status
# Statistics - MACFF - Statistic
# Statistics - MACFF - Status
# Statistics - G.bond
# Statistics - DSL Status
# Statistics - OAM
# Statistics - PPPoE
# Statistics - 802.1x Port Status
# Statistics - LLDP Status - Local Status
# Statistics - LLDP Status - Remote Status
# Statistics - LLDP Status - Statistic
# Switch - Switch Setup - DSCP # -> Switch - DSCP
# Switch - LLDP - Configuration
# Switch - LLDP - Basic TLV Setting
# Switch - LLDP - Org-specific TLV Setting
# VLAN - GE/GENNI VLAN Setup - VLAN Table #? VLAN - VLAN Table #?
# VLAN - GE/GENNI VLAN Setup - VLAN Untag and Copy-Cpbit #->VLAN - VLAN Untag and Copy-cpbit
# VLAN - Port VLAN TLS
# VLAN - Port VLAN Trunk
# VLAN - Port VLAN Translation


###
# date1 = str(datetime.datetime.now()).split(":")[0]
# date2 = str(datetime.datetime.now()).split(":")[1]
# file_name = f"Output\Web_EMS_{emsversion}_{date1}{date2}.txt"
# test_report = open(file_name,"w+")
from ems_test_module import *
###




In [ ]:
list1_o = ['4,9,16,MSC1240XA', '1,15,39,GLC1540X-55A', '2,17,16,OLC3416B-IA']
list1 = '1,15,39,GLC1540X-55A'


In [ ]:
ems.VLAN_VLAN_Table(driver,url,list1,username,password,file_name)    

In [ ]:
# Switch
# ems.Switch_DSCP(driver,url,list1,username,password,file_name)                               # case                                v
# ems.Switch_Storm_Control(driver,url,list1,username,password,file_name)                      # case                                  v

# # Port
# ems.Port_GENNI(driver,url,list1,MSC_slot,msc_port,username,password,file_name)                      # case                          v
# ems.Port_GE(driver,url,list1,MSC_slot,username,password,file_name)                      # case                                      v
# ems.Port_Qos(driver,url,list1,username,password,file_name)                           # undone  X   case                             v     un
ems.Port_DHCP(driver,url,list1,username,password,file_name)                     # case                                              x

# VLAN
ems.VLAN_VLAN_Table(driver,url,list1,username,password,file_name)                  # case                                           x
# ems.VLAN_Untag_And_Copy_Cpbit(driver,url,list1,username,password,file_name)        # case                                           v

In [ ]:
import ems_port_module as e_por
e_por.DHCP_Snooping_Setting_Priority_VLAN(driver,"4092,4093,4094")

In [ ]:
port_dhcp_ipmacbinding1_select
port_dhcp_ipmacbinding_select

In [ ]:
from ems_test_module import *
EMS_Provision_Config_path(driver,url)

In [ ]:
a = str(1)
b = '1'
print(type(a))
print(type(b))

In [ ]:
driver.get('http://192.168.169.86/global/rpindex2.msp')

In [ ]:
# EMS website > login
driver.get(EMS_IP)
time.sleep(5)
mod.Login_EMS(driver,username,"Pqa1234!!")
time.sleep(5)    
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("Multicast").click()          
time.sleep(1)
driver.find_element_by_link_text("VLAN Mode").click()
time.sleep(3)

e_mul.VLAN_Mode_Select(driver,1)
e_mul.VLAN_Mode_Delete(driver)
print(f"             [Multicast - VLAN Mode] EMS msg:{alert_accept_text(driver)}",file = open(file_name, '+a'))
if_find = False
for index in range(driver.find_elements_by_class_name("zy_tr.multicast_vlanmode_vlan_row").__len__()):
    if(driver.find_elements_by_class_name("zy_td.multicast_vlanmode_vlan_vlan")[index].text == "1"):                        
        if_find = True
        print(f"             [Multicast - VLAN Mode] EMS VLAN 1 found Delete Test fail" )
        if_EMS_error = True
        break
if(if_find == False):
    print(f"             [Multicast - VLAN Mode] EMS VLAN 1 not found Delete Test pass" )    


In [ ]:
e_pro.Profile_Path(driver,"GE Template")
profile_name = ['!!!GE_template_vlan_trunk_xxxxx','!!!GE_template_vlan_tls_xxxxx','!!!GE_template_vlan_translation']



# MODIFY -------------------------------------------------------------------
if_error = False
print(f"             [Profile - GE Template] (Modify)",file = open(file_name, '+a'))

# prof.1 modify
print(f"             [Profile - GE Template] (prof.1 '{profile_name[0]}')",file = open(file_name, '+a'))
if(e_pro.Profile_Table_List_Edit_Name(driver,profile_name[0]) == False):
    print(f"             [Profile - GE Template] not found '{profile_name[0]}' in the first or last page",file = open(file_name, '+a'))
    if_error = True
else:   
    # Delete VLAN TLS
    e_pro.GE_Template_VLAN_TLS_Select(driver,'1')
    e_pro.GE_Template_VLAN_TLS_Delete(driver)
    # Delete VLAN Trunk : UNI VLAN
    if_find = True
    while(if_find == True):
        try:
            e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_Select(driver,'1')
            e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_Delete(driver)
        except:
            if_find = False

    e_pro.GE_Template_Auto_Nego(driver,False)
    e_pro.GE_Template_Flow_Ctrl(driver,False)
    e_pro.GE_Template_Speed(driver,'10G full-duplex')
    e_pro.GE_Template_MTU(driver,'512')
    # VLAN Untag and Copy-Cpbit
    e_pro.GE_Template_VLAN_Untag_And_Copy_Cpbit_PVID(driver,'1')
    e_pro.GE_Template_VLAN_Untag_And_Copy_Cpbit_Pbit(driver,'0')
    e_pro.GE_Template_VLAN_Untag_And_Copy_Cpbit_Frame_Type(driver,'All')
    e_pro.GE_Template_VLAN_Untag_And_Copy_Cpbit_Copy_Cpbit(driver,False)
    e_pro.GE_Template_VLAN_Untag_And_Copy_Cpbit_Outer_TPID(driver,False)
    # VLAN
    e_pro.GE_Template_VLAN_New(driver)
    e_pro.GE_Template_VLAN_New_VLAN_ID(driver,'1')            # add invalid test?
    e_pro.GE_Template_VLAN_New_Mode(driver,'Normal')
    e_pro.GE_Template_VLAN_New_Untag(driver,False)    
    e_pro.GE_Template_VLAN_New_OK(driver)    


    # VLAN Trunk : UNI-VLAN
    if_find = True                          # (clear table)
    while(if_find == True):
        try:
            e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_Select(driver,'1')
            e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_Delete(driver)
        except:
            if_find = False
    e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_New(driver)
    e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_New_Uni_VID(driver,'1')
    e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_New_SVID(driver,'1')
    e_pro.GE_Template_VLAN_Trunk_UNI_VLAN_New_OK(driver)



    # VLAN Trunk : UNI-untag  check maximum
    e_pro.GE_Template_VLAN_Trunk_Uni_UnTag_New(driver)

    msg = alert_accept_text(driver)
    if('Maximum' not in msg): # 'over limit' -> 'Maximum'
        print(f"             [Profile - GE Template] msg:{msg} VLAN Trunk : UNI-untag, Detect 'Maximum' Test Fail",file = open(file_name, '+a'))
        # if_error_new = True
        if_error = True
    else:
        print(f"             [Profile - GE Template] msg:{msg} VLAN Trunk : UNI-untag, Detect 'Maximum' Test Pass",file = open(file_name, '+a'))
    # // =>Check over limit
    
    
    
    
    
    
    
    # VLAN Trunk : UNI-untag  check maximum
    try:
        e_pro.GE_Template_VLAN_Trunk_Uni_UnTag_Select(driver,'1')

        try:
            e_pro.GE_Template_VLAN_Trunk_Uni_UnTag_Delete(driver)
        except:
            print(f"             [Profile - GE Template] VLAN Trunk Uni Untag Delete Fail",file = open(file_name, '+a'))

    except:
        if_error = True
        print(f"             [Profile - GE Template] VLAN Trunk Uni Untag Select Fail",file = open(file_name, '+a'))
        print(f"             [Profile - GE Template] VLAN Trunk Uni Untag Delete Fail",file = open(file_name, '+a'))
    
    

In [ ]:
e_pro.Profile_Path(driver,"GE Template")
driver.find_element(By.XPATH,'//*[@name="Name" and @class="dragaccept"]').click()
# //*[@id="profile_table_head"]/thead/tr/th[4]/a
# driver.find_element(By.XPATH,'//*[@id="profile_table_head"]/thead/tr/th[4]/a').click()
# //*[@id="profile_table_head"]/thead/tr/th[3]
# driver.find_element(By.XPATH,'//*[@id="profile_table_head"]/thead/tr/th[3]').click()



In [ ]:
# from ems_test_module import *

# print(f"EMS - Multicast - MVLAN Test\n",file = open(file_name, '+a')) 
# if_EMS_error = False
# if_FW_error = False

# # --------------------- EMS web start ---------------------
# # EMS - Multicast - MVLAN

# # EMS website > login
# driver.get(EMS_IP)
# time.sleep(5)
# mod.Login_EMS(driver,username,"Pqa1234!!")
# time.sleep(5)    
# EMS_Provision_Config_path(driver,url)
# time.sleep(2)    
# driver.find_element_by_link_text("Multicast").click()          
# time.sleep(1)
# driver.find_element_by_link_text("MVLAN").click()
# time.sleep(3)

# if_mvlan_apply = True     

# # try: 

# # valid input 1, 4094 test
# test_keys = ["1","4094"]
# for i in test_keys:
#     e_mul.MVLAN_MVLAN(driver,i)
#     e_mul.MVLAN_Pbit(driver, "7")
#     e_mul.MVLAN_Add(driver)
#     print(f"             [Multicast - MVLAN] EMS msg:{alert_accept_text(driver)}",file = open(file_name, '+a'))
#     if_find = False
#     for index in range(driver.find_elements_by_class_name("zy_tr.multicast_mvlan_mvlan_table_row").__len__()):
#         if(driver.find_elements_by_id("multicast_mvlan_mvlan1_label")[index].text == i):                        
#             if_find = True
#             print(f"             [Multicast - MVLAN] EMS MVLAN:{i} Test pass",file = open(file_name, '+a'))
#             if(driver.find_elements_by_id("multicast_mvlan_pbit1_label")[index].text == "7"):
#                 print(f"             [Multicast - MVLAN] EMS Pbit:7 Test pass",file = open(file_name, '+a'))
#             else:
#                 print(f"             [Multicast - MVLAN] EMS Pbit:7 Test fail",file = open(file_name, '+a'))
#                 if_EMS_error = False
#             break
#     if(if_find == False):
#         print(f"             [Multicast - MVLAN] EMS MVLAN:{i} Test fail",file = open(file_name, '+a'))      
#         if_EMS_error = True
#         if_mvlan_apply = False        
# # invalid input      
# test_keys = ["0","4095"]
# for i in test_keys:
#     e_mul.MVLAN_MVLAN(driver,i)
#     e_mul.MVLAN_Add(driver)
#     print(f"             [Multicast - MVLAN] EMS msg:{alert_accept_text(driver)}",file = open(file_name, '+a'))
#     if_find = False
#     for index in range(driver.find_elements_by_class_name("zy_tr.multicast_mvlan_mvlan_table_row").__len__()):
#         if(driver.find_elements_by_id("multicast_mvlan_mvlan1_label")[index].text == i):                        
#             if_find = True
#             print(f"             [Multicast - MVLAN] EMS MVLAN:{i} Test fail",file = open(file_name, '+a'))                
#             if_EMS_error = True
#             break
#     if(if_find == False):
#         print(f"             [Multicast - MVLAN] EMS MVLAN:{i} Test pass",file = open(file_name, '+a'))      
        
# e_mul.MVLAN_MVLAN(driver,"@")
# e_mul.MVLAN_Add(driver)
# print(f"             [Multicast - MVLAN] EMS msg:{alert_accept_text(driver)}",file = open(file_name, '+a'))
# if_find = False
# for index in range(driver.find_elements_by_class_name("zy_tr.multicast_mvlan_mvlan_table_row").__len__()):
#     if(driver.find_elements_by_id("multicast_mvlan_mvlan1_label")[index].text == "@"):                        
#         if_find = True
#         print(f"             [Multicast - MVLAN] EMS MVLAN:@ Test fail",file = open(file_name, '+a'))
#         if_EMS_error = True
#         break
# if(if_find == False):
#     print(f"             [Multicast - MVLAN] EMS MVLAN:@ Test pass",file = open(file_name, '+a'))      


# # --------------------- EMS web end -----------------------
# # --------------------- FW web start ----------------------
# # Multicast > MVLAN > MVLAN
# driver.get(url) 
# time.sleep(3)
# mod.Login(driver,username,password)      
# # 路徑導向 Multicast > MVLAN > MVLAN
# mod.Switch_Default(driver)
# mod.Page_Switch(driver,"Multicast")
# mod.SubPage_Switch(driver,"MVLAN")
# mod.Switch_Frame(driver)
# mod.SubPage_Switch(driver,"MVLAN")  

# try:
#     if(driver.find_elements_by_class_name("td-table-content.rpMvlan_displayMvlan")[0].get_attribute("textContent") != "1"):
#         print(f"             [Multicast - MVLAN - MVLAN] FW MVLAN:1 Test Fail",file = open(file_name, '+a'))
#         if_FW_error = True
#     if(driver.find_elements_by_class_name("td-table-content.rpMvlan_displayMvlan")[1].get_attribute("textContent") != "4094"):
#         print(f"             [Multicast - MVLAN - MVLAN] FW MVLAN:4094 Test Fail",file = open(file_name, '+a'))
#         if_FW_error = True
#     if(driver.find_elements_by_class_name("td-table-content.rpMvlan_displayPbit")[0].get_attribute("textContent") != "7" or driver.find_elements_by_class_name("td-table-content.rpMvlan_displayPbit")[1].get_attribute("textContent") != "7"):
#         print(f"             [Multicast - MVLAN - MVLAN] FW Pbit:7 Test Fail",file = open(file_name, '+a'))
#         if_FW_error = True
# except:        
#     print(f"             [Multicast - MVLAN - MVLAN] FW Apply Fail",file = open(file_name, '+a'))
#     if_FW_error = True

# # --------------------- FW web end ------------------------


# # except:
# #     if_EMS_error = True
# #     print(f"             [Multicast - MVLAN] test fail",file = open(file_name, '+a'))


# # EMS Result
# if(if_EMS_error == True):
#     print(f"             [Multicast - MVLAN] EMS web check fail",file = open(file_name, '+a'))
# else:
#     print(f"             [Multicast - MVLAN] EMS web check pass",file = open(file_name, '+a')) 

# # FW Result
# if(if_FW_error == True):
#     print(f"             [Multicast - MVLAN] FW web check fail",file = open(file_name, '+a'))
# else:
#     print(f"             [Multicast - MVLAN] FW web check pass",file = open(file_name, '+a')) 

In [ ]:
# driver.find_element_by_id("multicast_mvlan_pbit1_label")[index].text

In [ ]:
#    Provision > Node Configuration        
# ====================================

print(f"EMS - Node Configuration Test Start\n",file = open(file_name, '+a'))


# print("======================================================================================",file = open(file_name, '+a'))
# print("No Card Test",file = open(file_name, '+a'))
# print("======================================================================================",file = open(file_name, '+a'))
# # ems.EMS_test_all_NoCard(driver,url,list1,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)

# # # No Card
ems.EMS_ACL_DHCP_Remote_Backup(driver,url,MSC_card,MSC_slot,username,password,file_name)  #  
ems.EMS_ACL_PPPoE(driver,url,MSC_card,username,password,file_name)                        #      
ems.EMS_ACL_Multicast(driver,url,MSC_card,username,password,file_name)                    #    
ems.Multicast_IGMP_Setup(driver,url,MSC_card,username,password,file_name)                 # msc need_modify   X
ems.Multicast_VLAN_Mode(driver,url,MSC_card,username,password,file_name)                  # msc need_modify   X
ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name)      # msc
# # (Sys)                                                                                     # msc                                                                               
# ems.Sys_General_Setup(driver,url,MSC_card,username,password,file_name) 
# ems.Sys_IPv4_Setup(driver,url,MSC_card,username,password,file_name)
# Ip_setup_gateway = ems.Sys_IPv6_Setup(driver,url,MSC_card,username,password,file_name)
# ems.Sys_IP_Route(driver,url,MSC_card,username,password,file_name, Ip_setup_gateway)    
# ems.Switch_General(driver,url,MSC_card,username,password,file_name)                         # msc 
# ems.Switch_Dot3ad(driver,url,MSC_card,MSC_slot,msc_port,username,password,file_name)        # msc   undone
# ems.Switch_Isolation(driver,url,MSC_card,username,password,file_name)                       # msc 
# ems.VLAN_Tag_Mode_Setup(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc



print("======================================================================================",file = open(file_name, '+a'))
print("Card Test",file = open(file_name, '+a'))
print("======================================================================================",file = open(file_name, '+a'))
for i in range(list1.__len__()):

    list_element = list1[i]

    slot = list_element.split(",")[1]
    port = list_element.split(",")[2]
    cardtype = list_element.split(",")[3]
    print(f"             Now in case:{list_element[:1]},  Slot-Port:{slot}-{port}, {cardtype}",file = open(file_name, '+a'))
    print("======================================================================================",file = open(file_name, '+a'))
    ems.EMS_test_all_ACL(driver,url,list_element,MSC_card,MSC_slot,username,password,file_name,test_retry_num)
    ems.EMS_test_all_Multicast(driver,url,list_element,MSC_card,MSC_slot,username,password,file_name,test_retry_num)
    ems.EMS_test_all_Switch(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)
    ems.EMS_test_all_Port(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)
    ems.EMS_test_all_VLAN(driver,url,list_element,MSC_card,MSC_slot,msc_port,username,password,file_name,test_retry_num)



# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

# # ACL
# ems.EMS_ACL_Anti_MAC_Spoof(driver,url,list1,username,password,file_name)                  # V case
# ems.EMS_ACL_ARP_Inspection(driver,url,list1,username,password,file_name)                  #   case 
# ems.EMS_ACL_DHCP_L2Agent(driver,url,MSC_card,username,password,file_name)                 #   msc   => card?
# ems.EMS_ACL_MAC_Count(driver,url,list1,username,password,file_name)                       #   case
# ems.EMS_ACL_MAC_Filter(driver,url,list1,username,password,file_name)                      #   case 
# ems.EMS_ACL_OUI_Filter(driver,url,list1,username,password,file_name)                      #   case  undone
# ems.EMS_ACL_Packet_Filter(driver,url,list1,username,password,file_name)                   #   case

# # Multicast
# ems.Multicast_Port_Mode(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc need_modify          => card?
# ems.Multicast_Multicast_MAC(driver,url,list1,username,password,file_name)                 # case
# ems.Multicast_Multicast_IP(driver,url,list1,username,password,file_name)                  # case

# # Switch
# ems.Switch_DSCP(driver,url,list1,username,password,file_name)                               # case  
# ems.Switch_Storm_Control(driver,url,list1,username,password,file_name)                      # case   

# # Port
# ems.Port_GENNI(driver,url,list1,MSC_slot,msc_port,username,password,file_name)                      # case  
# ems.Port_GE(driver,url,list1,MSC_slot,username,password,file_name)                      # case  
# ems.Port_Qos(driver,url,list1,username,password,file_name)                           # undone  X   case
# ems.Port_DHCP(driver,url,list1,username,password,file_name)                     # case

# # VLAN
# ems.VLAN_VLAN_Table(driver,url,list1,username,password,file_name)                  # case
# ems.VLAN_Untag_And_Copy_Cpbit(driver,url,list1,username,password,file_name)        # case

# # No Card
# ems.EMS_ACL_DHCP_Remote_Backup(driver,url,MSC_card,MSC_slot,username,password,file_name)  #   msc
# ems.EMS_ACL_PPPoE(driver,url,MSC_card,username,password,file_name)                        #   msc  
# ems.EMS_ACL_Multicast(driver,url,MSC_card,username,password,file_name)                    #   msc
# ems.Multicast_IGMP_Setup(driver,url,MSC_card,username,password,file_name)                 # msc need_modify   X
# ems.Multicast_VLAN_Mode(driver,url,MSC_card,username,password,file_name)                  # msc need_modify   X
# ems.Multicast_MVLAN_And_MVLAN_Table(driver,url,MSC_card,username,password,file_name)      # msc
# # (Sys)                                                                                     # msc                                                                               
# ems.Sys_General_Setup(driver,url,MSC_card,username,password,file_name) 
# ems.Sys_IPv4_Setup(driver,url,MSC_card,username,password,file_name)
# Ip_setup_gateway = ems.Sys_IPv6_Setup(driver,url,MSC_card,username,password,file_name)
# ems.Sys_IP_Route(driver,url,MSC_card,username,password,file_name, Ip_setup_gateway)    
# ems.Switch_General(driver,url,MSC_card,username,password,file_name)                         # msc 
# ems.Switch_Dot3ad(driver,url,MSC_card,MSC_slot,msc_port,username,password,file_name)        # msc   undone
# ems.Switch_Isolation(driver,url,MSC_card,username,password,file_name)                       # msc 
# ems.VLAN_Tag_Mode_Setup(driver,url,MSC_card,MSC_slot,username,password,file_name)         # msc



In [ ]:
from selenium import webdriver
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
import time
import selenium_module as mod
from customdriver import CustomDriver


time_sleep_before_find_element = 0 # 0.1
driver = CustomDriver(time_sleep = time_sleep_before_find_element)

driver.get("https://192.168.128.8:9116/") 
time.sleep(5)
mod.Login_EMS(driver,"admin","Pqa1234!!")
time.sleep(5)
# driver.find_element('id','about').click()
# emsversion = driver.find_element('id','emsversion').text.replace("\n","")   #.text.split("\n")[0]
# e = driver.find_element('xpath','//*[@id="emsversion"]/text()[1]').text
# print(e)

In [ ]:
driver.get('https://192.168.128.8:9116/netatlasems/index#')


In [ ]:
index = 1024
p = str(index)[:1]
a = str(index)[-2:]
print(p)
print(a)

In [ ]:
driver.find_element('id','provision_select_ont_ip_checkbox').click()

In [ ]:
import ems_profile_module as e

# click 'Select ONT'
driver.find_element("xpath",'//*[@id="create_service_background"]/div[1]/div[3]/div[2]/input[1]').click()

# V
e.Checkbox_EMS(driver,'provision_select_ont_ip_checkbox',True)

# input ip
driver.find_element('id','provision_select_ont_ip_text').send_keys('192.168.128.86')


In [ ]:
# # [2] input 
# # / EMS - Provision - Service - ONT
# e_ser.Service_path(driver,"ONT")
# # / click New Serive
# e_ser.New_Service(driver)


# # 
# driver.find_element("xpath",'//*[@id="create_service_background"]/div[1]/div[3]/div[2]/input[1]').click()

# # / input val -> HOW???
# # ...
# # input sn, password registration id 
# # ...

# input('...')

# [3] input
e_ser.New_Service_Service_Config_Double_Click(driver)
e_ser.New_Service_Service_Config_Active(driver,True)
e_ser.New_Service_Service_Config_Description(driver,'model') # Excel model name
e_ser.New_Service_Service_Config_ONT_Template(driver,'ONT_Template_name')
# / click OK

In [ ]:
import logging

def print_(t,n,file=0,type_ = 0):
    # if(file != 0):
    #     print(t,n*2,file=file)
    # else:
    #     print(t,n*2)
    if(file != 0):          # will print in text file
        match type_:
            case 1:
                logging.info(t)
                print(t,file=file)
            case _:
                pass
    else:                   # simply print
        print(t)

def test(file_name):
    test_report = open(file_name,"w+")

    print_('ascxsa1',5,file= open(file_name, '+a'))
    print_('ascxsa2',3)

test('20240123.txt')
# print(type(open(file_name)))

In [ ]:
a = [8,7,6]
a.pop(0)
print(a)

In [ ]:

e = driver.find_element('xpath','//*[@id="EditTable"]/div[2]/table/tbody/tr/td[4]')
e.click()
e.click()
# e.clear()
# pyautogui.typewrite("123")
# e.send_keys('1234')

In [ ]:
e = driver.find_element('css selector','#EditTable > div.jexcel_content > table > tbody > tr > td:nth-child(2)')
driver.execute_script("arguments[0].click()")

In [ ]:
e = driver.find_element('css selector','#EditTable > div.jexcel_content > table > tbody > tr > td:nth-child(2)')
from selenium.webdriver.common.action_chains import ActionChains
ActionChains(driver).move_to_element(e).click().perform()
e.send_keys('567')

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [ ]:
# driver.switch_to.frame("")

In [ ]:
p = ['!E2E_Lan_SFU_1001_','!E2E_Lan_SFU_1107_','!E2E_Lan_SFU_1108_','!E2E_Lan_SFU_all_']
for i in p:
    print(i, i.split('_')[-2], i.split('_')[-2][-1])

In [ ]:
prof = [['!E2E_10M_',10000]]
for p in prof:
    print(p[0],p[1])

In [ ]:
for index in range(1,9): 
    print (index%8)

In [ ]:
ser_prof = ['!E2E_Lan_SFU_1001_','!E2E_Lan_SFU_1107_','!E2E_Lan_SFU_1108_','!E2E_Lan_SFU_all_','!E2E_Lan_SFU_108_',
                '!E2E_Lan_SFU_101_','!E2E_Lan_SFU_102_','!E2E_Lan_SFU_103_','!E2E_Lan_SFU_104_','!E2E_Lan_SFU_105_','!E2E_Lan_SFU_106_','!E2E_Lan_SFU_107_']

print(ser_prof[0:5])
print(ser_prof[5:])

In [ ]:


from selenium.webdriver.common.keys import Keys
# driver.find_element("xpath","//a[@title='New Service']").click()
x = '//*[@id="EditTable"]/div[2]/table/tbody/tr/td[2]'
x2 = "//td[@data-x='0']"
# driver.find_element("xpath",x).click()
ele = driver.find_element("xpath",x)
ele.click()
time.sleep(1)
# ele.send_keys('123')
# ele.send_keys(Keys.CONTROL + 'a')
driver.execute_script("arguments[0].setAttribute('value', '" + 'my_string123' +"')", ele)

In [ ]:
driver.find_element('xpath','//*[@value="Save"]').click()

In [ ]:
ele = driver.find_element("xpath",x2)
ele.click()
ele.click()
time.sleep(1)
# driver.execute_script("arguments[0].setAttribute('value', '" + 'my_string' +"')", ele)

In [ ]:
import urllib
src = driver.find_element('','').get_attribute('src')

urllib.urlretrieve(src, "captcha.png")

In [ ]:
o = [['A','1'],['B','2']]
for i in o:
    i.append(int(i[1])-1)

print(o)

In [ ]:
# ls = [1,2,1,3,1,5,2]
ls = [(179, 171),(179, 184),(179, 238)]
print(list(set(ls)))
print(ls[0][0])

In [ ]:
# o = [{}]
o = []
for i in  range(3):
    o.append({})
    # o[i]['a'] = 'A'
    # o[i]['1'] = f'11{str(i)}'

    ont_index = o.__len__()-1
    o[ont_index]['model'] = 'A'
    o[ont_index]['id'] = f'11{str(i)}'

print(o)

for ont in o:
    print(ont['id'])

print(o.__len__())

for index in range(o.__len__()):
    print(o[index]['id'],o[index]['model'])
    o[index]['ver'] = '+'
print(o)

#  id, model, ver

In [ ]:
o = [{"id":"1-1-1",'model':'A','ver':'5'},{"id":"1-1-2",'model':'B','ver':'10'}]
print(o)
for i in o:
    i['status'] = i['ver'] + '*'
print(o)

# exe upgrade ver
print('exe upgrade ver')
for ont in o:
    print('- ',ont['id'],ont['ver'])

# [Step4] Check ont-fw-upgrade result.
print('# [Step4] Check ont-fw-upgrade result.')
for ont in o:




a = {"1-1-1":{'model':'A','ver':'5'},"1-1-2":{'model':'B','ver':'10'}}
print(a)
for i in a:
    # i['status'] = i['ver'] + '*'
    # i.k
    print(i)

print(a['1-1-1'])

In [ ]:
from tttttemp import *

file_name = "tttttemp.txt"
output_txt = open(file_name,"w+")

ont_dict = {"tA":"Fail"}
ont_google_sheet(ont_dict,file_name)

In [ ]:
import json
import re

def check_nested_keys(data,key):
    if(isinstance(data,dict)):
        if(key in data):
            print(f'found key {key} {data[key]}')
        else:
            for subkey, val in data.items():
                if(isinstance(val,dict)):
                    check_nested_keys(val, key)
    elif(isinstance(data,list)):   
        for item in data:
            check_nested_keys(item,key)

    # find a certain key by recursive

def check_nested_keys_2(data,key,hierarchy):
    
    if(isinstance(data,dict)):
        print(' in dict')
        if(all(level in data for level in hierarchy)):
            nested_dict = data
            for level in hierarchy:
                nested_dict = nested_dict[level]
            if(key in nested_dict):
                print(f'found key {key} {nested_dict[key]}')
        for subkey, val in data.items():
            check_nested_keys_2(val,key,hierarchy)
    elif(isinstance(data,list)):   
        print(' in list')
        for item in data:
            check_nested_keys_2(item,key,hierarchy)

def check_k(data, my_list):
    print(my_list)
    key = my_list.pop()    
    print(key)
    print(my_list)
    check_nested_keys_2(data,key,my_list)

# def check_speci_path(data,my_list):
#     for key in my_list:
#         pass

def j_check(content,value,string,d):
    if(content not in d):
        return "Not found"
    blank = " "
    if(content == value):   
        print(f'{blank}{string} Pass ({value})')
        return True
    else:
        print(f"{blank}{string} Fail ({content} != {value})")
        return False


JSON_file = 'wanip.json'

with open(JSON_file) as f:
    data = json.load(f,strict=False)
    d = json.dumps(data)


s = ["Services","VoiceService","0","PhyInterface"]
check_k(data,s)

In [ ]:
s = "Services VoiceService 0 PhyInterface"
### 用 list 存?
s = ["Services","VoiceService","0","PhyInterface"]
print(s)

In [ ]:
print(data['Services']['VoiceService'][0]) 
a = data['Services']['VoiceService'][5]

In [ ]:
print(d)

In [ ]:
dict1 = {"normal":"0", "1plus1":"1"}
temp = "2"
a = dict1.keys()["0"]
print(a)

In [ ]:
my_list = [[57,193],[82,193]]
my_list[0]

In [ ]:
# driver = webdriver.Chrome()
# from webdriver_manager.chrome import ChromeDriverManager 
# driver = webdriver.Chrome(ChromeDriverManager().install())

In [ ]:
# # # (Max Profile) mini EMS WEB test environment
# # ======================================  
# from selenium.webdriver.common.by import By
# import ems_vlan_module as e_vla
# import ems_profile_module as e_pro
# import ems_test_module as ems
# import selenium_module as mod
# import time
# from selenium import webdriver


# def Test2_find_profile_name(driver):
#     driver.switch_to.default_content()
#     time.sleep(1) 
#     driver.find_element(By.LINK_TEXT,"Provision").click()
#     time.sleep(1)
#     driver.find_element(By.LINK_TEXT,"Profile").click()
#     time.sleep(1)

#     e_pro.Profile_Select(driver,"ONT Service Profile")
#     time.sleep(1)

#     # for index in 
#     # e_pro.Edit(driver,index)


# def Test3_max_profile(driver,profile_test_name='ONT Security Profile'):
    
#     # rewrite --------------------------
#     print(f'Now in {profile_test_name} \n Processing...')

#     if_error = False

#     MAX_MSG = 'Reach the maximum number of profiles 3000 in database'
    
#     driver.find_element(By.LINK_TEXT,"Profile").click()
#     time.sleep(1)
#     e_pro.Profile_Select(driver,profile_test_name)
#     time.sleep(1)

#     # limit_num = 3011
#     limit_num = 2000
#     for index in range(1,limit_num):
#         e_pro.New_Profile(driver)
#         e_pro.Profile_Name(driver,f'Auto_max_{index}')
#         e_pro.Save(driver)
#         msg = ems.alert_accept_text(driver)
#         if('uccess' not in msg):
#             print(f'msg:{msg}')
#             if(MAX_MSG not in msg):
#                 if_error = True
#             break

#     # Check 3000    
#     # reload page
#     driver.find_element(By.LINK_TEXT,"Provision").click()
#     time.sleep(1)
#     driver.find_element(By.LINK_TEXT,"Profile").click()
#     time.sleep(1)
#     e_pro.Profile_Select(driver,profile_test_name)
#     time.sleep(1)
#     # check total data num
#     total_d = driver.find_element(By.ID,'table_total_data_profile_table').text
#     print(f"web show total data is:{total_d}")
#     if('3000' not in total_d):
#         print('max 3000 test fail')
#         if_error = True
#     else:
#         print('max 3000 test pass')

#     if(if_error == True):
#         print('Max Test Fail')
#     else:
#         print('Max Test Pass')

# def Test3_max_profile_2000(driver,profile_test_name,limit_num=2000,start_n=1):
    
#     # rewrite --------------------------
#     print(f'Now in {profile_test_name} \n Processing...')

#     if_error = False
    
#     driver.find_element(By.LINK_TEXT,"Profile").click()
#     time.sleep(1)
#     e_pro.Profile_Select(driver,profile_test_name)
#     time.sleep(1)

#     # limit_num = 3011
#     # limit_num = 2000
#     limit_num = limit_num + 1
#     for index in range(start_n,limit_num):
#         e_pro.New_Profile(driver)
#         e_pro.Profile_Name(driver,f'Auto_max_{index}')
#         e_pro.Save(driver)
#         try:
#             msg = ems.alert_accept_text_no_block(driver)
#             if('uccess' not in msg):
#                 print(f'msg:{msg}')
                
#                 break
#         except:
#             print(f'{index} may save fail')
#             pass

#     print('finished.')
    

    

# # MAIN ----------------------------------------------------------------------

# driver = webdriver.Chrome()
# driver.get("http://192.168.128.8:9116/")
# time.sleep(5)
# mod.Login_EMS(driver,"admin","Pqa1234!!")
# time.sleep(5)

# driver.switch_to.default_content()
# time.sleep(1) 
# driver.find_element(By.LINK_TEXT,"Provision").click()
# time.sleep(1)

# # ONT Security Profile  # 3000

# # Test3_max_profile(driver,'ONT Service Profile')
# # print()
# # Test3_max_profile(driver,'ONT QoS bandwidth Profile')
# # print()
# # Test3_max_profile(driver,'ONT Multicast Profile')

# # 2000
# # Test3_max_profile_2000(driver,'IGMP Group Privilege Profile',limit_num=2000,start_n=200) 
# # print()
# # Test3_max_profile_2000(driver,'Shaping Profile',limit_num=2000,start_n=200) 
# # print()

# # Test3_max_profile_2000(driver,'Rate Limit Profile',limit_num=2000,start_n=200) 
# # print()

# # Test3_max_profile_2000(driver,'Weight Profile',limit_num=2000,start_n=314) 
# # print() 
# Test3_max_profile_2000(driver,'GE Template',limit_num=2000,start_n=563) # stop at 1957?
  

# ##### find_element()   will wait time.sleep(0.7) ->  time.sleep(0)now

In [ ]:
# # (Debug) mini FW Web test environment
# # ======================================  

# from os import times
# import time
# from turtle import Turtle
# from xml.etree.ElementTree import ProcessingInstruction
# from attr import field
# from pyrsistent import s
# import selenium_module as mod
# import acl_module as acl
# import alarm_module as ala
# import diagnostic_module as dia
# import line_card_module as lc
# import l3_routing_module as l3r
# import maintenance_module as mai
# import multicast_module as mul
# import port_module as port
# import profile_module as pro
# import statistics_module as sta
# import switch_module as swi
# import sys_module as sys
# import vlan_module as vlan
# import config_save_module as con
# from selenium.webdriver.support.ui import Select
# import pyautogui
# import datetime

# from selenium import webdriver
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.common.by import By
# # import ems_vlan_module as e_vla
# # import ems_test_module as ems
# import selenium_module as mod
# import time

# def IN_Web(driver):
#     driver.get('http://192.168.169.86') 
#     time.sleep(3)
#     mod.Login(driver,"admin","1234")  
#     try:
#         driver.find_element(By.XPATH,"/html/body/form/table/tbody/tr[5]/td[2]/input").click()
#     except:
#         pass   
#     time.sleep(2)

# # ========================================================================================================================


# driver = webdriver.Chrome()
# IN_Web(driver)

# # --- Start ----- -- -- -- -- -- -- -- -- -- -- -- -- --  -- -- -- -- -- --  -- -- -- -- -- --  -- -- -- -- -- --  -- -- -- --   -- -- -- 


# try:

#     # VLAN Port VLAN Trunk UNI-untag
#     # VLAN > Port VLAN Trunk > UNI-untag
#     # -------------------------start-------------------------

#     if_error = False

#     # 路徑導向 VLAN > Port VLAN Trunk > UNI-untag
#     mod.Switch_Default(driver)
#     mod.Page_Switch(driver,"VLAN")
#     mod.SubPage_Switch(driver,"Port VLAN Trunk")
#     mod.Switch_Frame(driver)
#     time.sleep(1)
#     mod.Tab_Switch(driver,"UNI-untag")

#     # slot select
#     if_load = False
#     while(if_load == False):
#         try:
#             driver.find_element_by_xpath(f"//*[@name='lctbl_selSlot']/option[2]")
#             vlan.Slot_Select(driver,glc_slot)
#             vlan.Port_Select(driver,glc_port)
#             vlan.Load(driver)
#             if_load = True
#         except:
#             time.sleep(5)
#             vlan.Load(driver)
#             time.sleep(1)
#             if_load = False
    
#     # VLAN Port VLAN Trunk UNI-untag Test
#     vlan.Port_VLAN_Trunk_UNI_untag_Subnet_IP(driver,True)
#     vlan.Port_VLAN_Trunk_UNI_untag_Subnet(driver,"10.10.10.1/24")
#     vlan.Port_VLAN_Trunk_UNI_untag_SVID(driver,"2")
#     vlan.Port_VLAN_Trunk_UNI_untag_Spbit(driver,"2")
#     vlan.Port_VLAN_Trunk_UNI_untag_Apply(driver)
#     # pyautogui.press("enter")
#     WebDriverWait(driver, 10).until(EC.alert_is_present()).accept()
#     for test_port in range(40):
#         vlan.Load(driver)
#         time.sleep(1)
#         vlan.Port_Select(driver,test_port+1)
#         time.sleep(0.5)
#         vlan.Load(driver)
#         vlan.Port_VLAN_Trunk_UNI_untag_Ether_Type(driver,True)
#         vlan.Port_VLAN_Trunk_UNI_untag_Ether_Type_Input(driver,"ffff")
#         vlan.Port_VLAN_Trunk_UNI_untag_SVID(driver,"1")
#         vlan.Port_VLAN_Trunk_UNI_untag_Spbit(driver,"1")
#         vlan.Port_VLAN_Trunk_UNI_untag_Apply(driver)
#         vlan.Port_VLAN_Trunk_UNI_untag_SVID(driver,"4094")
#         vlan.Port_VLAN_Trunk_UNI_untag_Spbit(driver,"7")
#         vlan.Port_VLAN_Trunk_UNI_untag_CVID(driver,"4094")
#         vlan.Port_VLAN_Trunk_UNI_untag_Apply(driver)
#         if(driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_type")[0].get_attribute("textContent") != "trunk" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_svid")[0].get_attribute("textContent") != "4094" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_spbit")[0].get_attribute("textContent") != "7" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_cvid")[0].get_attribute("textContent") != "4094" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_ether_type")[0].get_attribute("textContent") != "-" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_HideSubnet.rpVlanTrunkUniUntag_config_table_data_subnet_ip")[0].get_attribute("textContent") != "-"):
#             print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Port{test_port+1} trunk Apply Test Fail",file = open(file_name, '+a'))
#             if_error = True
#         if(driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_type")[1].get_attribute("textContent") != "protocol" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_svid")[1].get_attribute("textContent") != "1" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_spbit")[1].get_attribute("textContent") != "1" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_cvid")[1].get_attribute("textContent") != "-" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_ether_type")[1].get_attribute("textContent") != "0xffff" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_HideSubnet.rpVlanTrunkUniUntag_config_table_data_subnet_ip")[1].get_attribute("textContent") != "-"):
#             print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Port{test_port+1} protocol Test Fail",file = open(file_name, '+a'))
#             if_error = True
#         if(driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_type")[2].get_attribute("textContent") != "subnet" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_svid")[2].get_attribute("textContent") != "2" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_spbit")[2].get_attribute("textContent") != "2" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_cvid")[2].get_attribute("textContent") != "-" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_config_table_data_ether_type")[2].get_attribute("textContent") != "-" or driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntag_HideSubnet.rpVlanTrunkUniUntag_config_table_data_subnet_ip")[2].get_attribute("textContent") != "10.10.10.1/24"):
#             print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Port{test_port+1} subnet Test Fail",file = open(file_name, '+a'))
#             if_error = True
    
#     # Slot Select Test
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Select(driver,glc_slot)
#     if(driver.find_element_by_id("dataTotal").get_attribute("textContent") != "showing 1 ~ 40, total 120"):
#         print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Total:120 Test Fail",file = open(file_name, '+a'))
#         if_error = True
#     else:
#         vlan.Port_VLAN_Trunk_UNI_untag_Slot_Next(driver)
#         driver.find_element_by_name("rpVlanTrunkUntagPort_sltEntry").click()
#         vlan.Port_VLAN_Trunk_UNI_untag_Slot_Delete(driver)
#         if(driver.find_element_by_id("dataTotal").get_attribute("textContent") != "showing 1 ~ 40, total 119"):
#             print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Delete 1 Test Fail",file = open(file_name, '+a'))
#             if_error = True
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Next(driver)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Select_All(driver,True)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Delete(driver)
#     # pyautogui.press("enter")
#     WebDriverWait(driver, 10).until(EC.alert_is_present()).accept()
#     time.sleep(1)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Next(driver)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Select_All(driver,True)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Delete(driver)
#     # pyautogui.press("enter")
#     WebDriverWait(driver, 10).until(EC.alert_is_present()).accept()
#     time.sleep(1)
#     try:
#         for index in range(40):
#             if(driver.find_elements_by_class_name("td-table-content.rpVlanTrunkUniUntagPort_type")[index].get_attribute("textContent") == "protocol"):
#                 driver.find_elements_by_name("rpVlanTrunkUntagPort_sltEntry")[index].click()
#         vlan.Port_VLAN_Trunk_UNI_untag_Slot_Delete(driver)
#         time.sleep(1)
#     except:
#         pass
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Select_All(driver,True)
#     vlan.Port_VLAN_Trunk_UNI_untag_Slot_Delete(driver)
#     time.sleep(1)
#     if(driver.find_element_by_id("dataTotal").get_attribute("textContent") != "showing 0 ~ 0, total 0"):
#         print(f"             [VLAN - Port VLAN Trunk - UNI-untag] Delete All Test Fail",file = open(file_name, '+a'))
#         if_error = True

#     # VLAN Port VLAN Trunk UNI-untag Result
#     if(if_error == False):
#         print(f"[MSAN1-11995][VLAN - Port VLAN Trunk - UNI-untag] Result Pass",file = open(file_name, '+a'))
#     else:
#         print(f"[MSAN1-11995][VLAN - Port VLAN Trunk - UNI-untag] Result Fail",file = open(file_name, '+a'))

# except Exception as e: 
#     print(str(e).split("\n")[0])
#     print(e)



In [ ]:
# #    (Debug) mini EMS WEB test environment:  Provision > Profile        
# # =================================================================

# simple execute environ (can just execute this cell code )

from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
import videocap_module as videocap
import ems_test_module as ems
import selenium_module as mod
import time
import datetime
from selenium.webdriver.common.by import By
import ems_profile_module as e_pro

from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
# pip install webdriver-manager
# driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install())) 
# driver = webdriver.Chrome()
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))


username = "admin"
ems_password = "Pqa1234!!"
date1 = str(datetime.datetime.now()).split(":")[0]
date2 = str(datetime.datetime.now()).split(":")[1]
try:
    old_n = file_name
except:
    pass
file_name_1 = f"simple_temp_test_{date1}{date2}.txt"
file_name = file_name_1
simple_test_report = open(file_name_1,"w+")

start_time = time.time()

# # video
# video_file = 'data_profile_qosbdw_valid_alert.mp4'
# t1 = videocap.VideoCapThread(viseo_file)
# t1.start()

# # -- Test Start ---------------------------------------------------------------------------- #

list1 =''
MSC_card =''
MSC_slot =''
# ems.EMS_test_all_Profile(driver,url,list1,MSC_card,MSC_slot,username,ems_password,file_name,test_retry_num_1 = 1)

#                                                                              A  (already add, only input no check)
#                                                                             -V  (wait for exe or modify) --V need modify
#                                                                              V  (done)

print(f"EMS - Profile Test Start\n",file = open(file_name, '+a'))
# driver.fullscreen_window()
# try:

# ems.IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)
# ems.Retry_Test_func(ems.ONT_UNI_Profile(driver,username,ems_password,file_name),file_name,test_limit=2)
    # ems.ONT_Service_Profile(driver,username,ems_password,file_name)              # V  V
# ems.ONT_QoS_Bandwidth_Profile(driver,username,ems_password,file_name)      # V
    # ems.ONT_Multicast_Profile(driver,username,ems_password,file_name)          # -V
# ems.ONT_UNI_Profile(driver,username,ems_password,file_name)                # -V
# ems.ONT_Security_Profile(driver,username,ems_password,file_name)           # V
    # ems.ONT_ONT_Profile(driver,username,ems_password,file_name)                # A
    # ems.ONT_VoIP_SIP_Profile(driver,username,ems_password,file_name)           # -V
# ems.ONT_VoIP_Common_Profile(driver,username,ems_password,file_name)        # V
    # ems.ONT_VoIP_Dial_Plan_Profile(driver,username,ems_password,file_name)     # V
    # ems.ONT_ACL_Profile(driver,username,ems_password,file_name)                # V 
# ems.ONT_Alarm_Profile(driver,username,ems_password,file_name)              # V

    # ems.ONT_Template_Profile(driver,username,ems_password,file_name)           # A

# ems.Retry_Test_func(ems.IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name),file_name,test_limit=2)
# ems.IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)           # -V
    # ems.Shaping_Profile(driver,username,ems_password,file_name)                        # -V
# ems.Weight_Profile(driver,username,ems_password,file_name)                         # -V
# ems.Rate_Limit_Profile(driver,username,ems_password,file_name)                     # -V
ems.GE_Template(driver,username,ems_password,file_name)                            # -V

    # ems.Create_IGMP_Group_Privilege_Profile(driver,username,ems_password,file_name)


# def Test_f(func):
#     try:
#         func
#     except:
#         print('in except')

# ems.ONT_UNI_Profile(driver,username,ems_password,file_name)
# Test_f(ems.ONT_UNI_Profile(driver,username,ems_password,file_name))

# except Exception as e:    
#     driver.save_screenshot('./ActionFail_image.png')
#     print('VSCode: ',str(e).split('\n')[0],file = open(file_name, '+a'))

# ems.Profile_Mode_Page_Test(driver,'000',prof_name,file_name):                   


print(f"EMS - Profile Test End\n",file = open(file_name, '+a'))

# # -- Test End   ---------------------------------------------------------------------------- #


# t1.stop_record()

# end_t = time.time() 
# print(f'{round((end_t - start_t),2)} sec')
# print(f'{round((end_t - start_t)/60,2)} min')
end_time = time.time() 
total_time = end_time - start_time
print(f'Total time: {int(total_time/60/60)} hr {int(total_time/60%60)} min {int(total_time%60)} sec',file = open(file_name, '+a'))

# simple_test_report.close()

try:
    file_name = old_n
except:
    pass

In [ ]:
driver.quit()

In [ ]:
from selenium.webdriver.support.ui import Select

a = Select(driver.find_element(By.ID,f"ge_template_igmp_combobox")).first_selected_option
print(a)
print(type(a))

if(a == 'No options are selected'):
    print('pass')

In [ ]:
if(driver.find_element(By.ID,f"ge_template_groupenable_checkbox").is_selected() != False):
    print(f"             [Profile - GE Template] IGMP Group Privilege Profile checkbox:X Test Fail")
    if_error = True 
else:
    print(f"             [Profile - GE Template] IGMP Group Privilege Profile checkbox:X Test Pass")


In [ ]:
len = driver.find_elements('class name','zy_tr.ge_template_vlan_table_row').__len__()
print(len)
if(len):
    print('iii')

In [ ]:
# unselect IGMP prof TRY


from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
# ele = driver.find_element(By.XPATH,'//*[@id="ge_template_igmp_combobox"]/option[2116]')


# ele.click()
# ele.send_keys(Keys.CONTROL)

p = '1234567890123456789012345678001'

Select(driver.find_element(By.ID,f"ge_template_igmp_combobox")).select_by_visible_text(p)

In [ ]:
Select(driver.find_element(By.ID,f"ge_template_igmp_combobox")).deselect_by_visible_text(p)

In [ ]:
driver.get('http://192.168.169.86')

In [ ]:
driver.switch_to.default_content()

driver.switch_to.frame(0)


len = driver.find_elements('xpath','//*[@class="td-table-content"]').__len__()
print(len)
print(len/3)

get_index = driver.find_elements('xpath','//*[@class="td-table-content"]')[len-3].text
print(get_index)

# get_index = driver.find_elements('xpath','//*[@class="td-table-content"]')[len-6].text
# print(get_index)

# driver.find_elements('xpath','//*[@class="table-sample-4"]/tbody/tr[2]/td[1]')[ ].text

In [ ]:
e_pro.Uni_Profile_New_Lan_Filter_untag(driver,6)

In [ ]:

from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.chrome.service import Service as ChromeService 
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))

In [ ]:
def Test_f(func):
    
    func()
    # try:
    #     func
    #     print('in try')
    # except:
    #     print('in except')
    

def f():
    print('i am f')
#     raise Exception
#     raiser()

# def raiser():
#     if (False):
#         print('raiser: f')
#         pass
#     else:
#         print('raiser: r')
#         raise

Test_f(f)
# f()


# -----------------------------------
# def raiser():
#     if False: 
#         pass
#     else:
#         raise ValueError

# try:
#     raiser()
# except ValueError:
#     print ("caught")

In [ ]:
def test ():
   print ("test was invoked")

def invoker(func):
   func()

invoker(test)

In [ ]:
driver.find_elements_by_id("rpXGSponOntProf_wifi2_ssid")[index-1].clear()

In [ ]:
 print(driver.find_element(By.XPATH,f'//*[@id="ge_igmp_enable_checkbox16"]').is_selected())

In [ ]:
from selenium.webdriver.support.ui import Select
index = 1
print(Select(driver.find_element(By.ID,f"ge_igmp_privilege_combobox{index}")).first_selected_option.get_attribute("value"))

In [ ]:
import ems_profile_module as e_pro

In [ ]:
import videocap_module as videocap

In [ ]:
from selenium.common.exceptions import InvalidElementStateException

In [ ]:
e = 'sdavav\nkaowe'
print(f'e:{e}')
e = str(e).split('\n')[0]
print(f"VSCode: {e}")

try:
    a = 0/0
except Exception as e:
    print(f'e:{e}')
    e = str(e).split('\n')[0]
    print(f"VSCode: {e}")

In [ ]:
try:
    # ...
    pass
except NoSuchElementException:
    raise Exception
except:
    # ...
    pass


In [ ]:
file_name = f"ytest.txt"
simple_test_report = open(file_name,"w+")

def F(**kwargs):
    print(kwargs['n'],file = open(file_name, '+a'))
    try:
        n = kwargs['n']
        m = kwargs['m']
        print(kwargs['n'],file = open(file_name, '+a'))
        print(f"{kwargs['n']},{kwargs['m']}",file = open(file_name, '+a'))
        print(n,m,file = open(file_name, '+a'))
        print(n*m/0,file = open(file_name, '+a'))
        print('in try',file = open(file_name, '+a'))
    except:
        print('in except',file = open(file_name, '+a'))

    print('F() end',file = open(file_name, '+a'))

def MainF(func):
    func


# ===SRART=====

MainF(F(n=1,m=0))

In [ ]:
from selenium import webdriver
driver = webdriver.Chrome()
driver.get('http://192.168.128.8:9116/')

In [ ]:
import ems_test_module as ems
import ems_profile_module as e_pro
import time

In [ ]:
raise Exception

In [ ]:
##### try hide browser
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# driver_exe = 'chromedriver'
options = Options()
options.add_argument("--headless")
# driver = webdriver.Chrome(driver_exe, options=options)
driver = webdriver.Chrome(options=options)
 

In [ ]:
from selenium import webdriver
driver = webdriver.Chrome()
driver.get('http://192.168.169.86')



In [ ]:
username = 'admin'
password = '1234'
username_ = driver.find_element_by_name("user")
username_.clear()
username_.send_keys(username)
password_ = driver.find_element_by_name("pass")
password_.clear()
password_.send_keys(password)
driver.find_element('xpath',f'//*[@value="Login"]').click()

In [ ]:
%tb

In [ ]:
from selenium import webdriver
import videocap_module as videocap
import ems_test_module as ems
import selenium_module as mod
import time
import datetime
from selenium.webdriver.common.by import By
import ems_profile_module as e_pro

from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
# pip install webdriver-manager
# driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install())) 
driver = webdriver.Chrome()

In [ ]:
driver.get('http://192.168.128.8:9116/')

In [ ]:
print(driver.find_element(By.ID,"ont_multicast_vid_text").get_attribute("value"))

In [ ]:
# ========================================================================================================================

In [ ]:
pip install pypiwin32

In [ ]:
for i in range(10):
    print(i)
    if(i==5):
        break

In [ ]:
def chrtonum(str): #(A~Z)
    # return int(ord(chr))-64

    # process each character
    # c = str.__len__() - 1
    # num = int(ord(str[-1]))-64 + c*26
    print(1)

    num = 0
    for i in range (str.__len__()):
        x = str[str.__len__()-1-i]
        x = int(ord(str[-1]))-64
        # print(x)
        num += x * (26 * i)

    return num
print(1)
print(chrtonum("AA"))


In [ ]:
print(26**2)

In [ ]:
my_dict = {"name":"A","year":1234}
print(my_dict.items())  # dict_items([('name', 'A'), ('year', 1234)])
print(my_dict.keys())   # dict_keys(['name', 'year'])
print(my_dict['name'])  # A

In [ ]:
import win32com.client as win32
outlook = win32.Dispatch('outlook.application')
mapi = outlook.GetNamespace('MAPI')

for account in mapi.Accounts:
    myaccount = account.DeliveryStore.DisplayName
    print(myaccount)

# for idx, folder in enumerate(mapi.Folders(myaccount).Folders):
#     print(idx+1,folder)


In [ ]:
for idx, folder in enumerate(mapi.GetDefaultFolder(6).Folders):
    print(idx+1,folder)

In [ ]:
# msg = mapi.Folders(myaccount).Folders("收件匣").Items

file_name = f"subject_firmware release.txt"
simple_test_report = open(file_name,"w+")

msgs = mapi.Folders(myaccount).Folders(2).Items
for msg in list(msgs):    
    subj = msg.Subject
    if('Firmware Release' in subj):
        print(f"{subj}",file = open(file_name, '+a'))
    # print(msg.Subject)